# Backtrader Quick Start Tutorial

This interactive tutorial will guide you through creating your first trading strategy with Backtrader.

## What You'll Learn

1. Installing Backtrader
2. Loading data
3. Creating a simple strategy
4. Running a backtest
5. Analyzing results

## 1. Installation

First, make sure Backtrader is installed:

In [ ]:
# Install Backtrader (uncomment if needed)
# !pip install backtrader

import backtrader as bt
import datetime
import pandas as pd

print(f"Backtrader version: {bt.__version__}")

## 2. Create Sample Data

For this tutorial, we'll create some sample price data:

In [ ]:
# Create sample data
import numpy as np

dates = pd.date_range('2023-01-01', periods=100, freq='D')
prices = 100 + np.cumsum(np.random.randn(100) * 2)

data = pd.DataFrame({
    'open': prices + np.random.randn(100) * 0.5,
    'high': prices + abs(np.random.randn(100) * 1),
    'low': prices - abs(np.random.randn(100) * 1),
    'close': prices,
    'volume': np.random.randint(1000, 10000, 100)
}, index=dates)

# Save to CSV
data.to_csv('sample_data.csv')
print("Sample data created:")
print(data.head())

## 3. Define a Simple Strategy

Let's create a simple moving average crossover strategy:

In [ ]:
class SmaCrossStrategy(bt.Strategy):
    """Simple Moving Average Crossover Strategy.
    
    Buy when fast MA crosses above slow MA.
    Sell when fast MA crosses below slow MA.
    """
    
    params = (
        ('fast_period', 10),
        ('slow_period', 30),
    )
    
    def __init__(self):
        # Create moving averages
        self.fast_ma = bt.indicators.SMA(self.data.close, period=self.p.fast_period)
        self.slow_ma = bt.indicators.SMA(self.data.close, period=self.p.slow_period)
        
        # Create crossover signal
        self.crossover = bt.indicators.CrossOver(self.fast_ma, self.slow_ma)
    
    def next(self):
        if not self.position:  # Not in market
            if self.crossover > 0:  # Fast MA crossed above slow MA
                self.buy()
        else:  # In market
            if self.crossover < 0:  # Fast MA crossed below slow MA
                self.sell()
    
    def notify_order(self, order):
        if order.status in [order.Completed]:
            if order.isbuy():
                print(f'BUY executed: Price={order.executed.price:.2f}')
            elif order.issell():
                print(f'SELL executed: Price={order.executed.price:.2f}')

print("Strategy defined successfully!")

## 4. Set Up and Run Backtest

Now let's create a Cerebro instance and run the backtest:

In [ ]:
# Create Cerebro instance
cerebro = bt.Cerebro()

# Add data feed
data_feed = bt.feeds.GenericCSVData(
    dataname='sample_data.csv',
    datetime=0,
    open=1,
    high=2,
    low=3,
    close=4,
    volume=5,
    dtformat='%Y-%m-%d',
)
cerebro.adddata(data_feed)

# Add strategy
cerebro.addstrategy(SmaCrossStrategy)

# Set initial cash
cerebro.broker.setcash(10000.0)

# Set commission
cerebro.broker.setcommission(commission=0.001)

# Add analyzers
cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')
cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')

# Print starting conditions
print(f'Starting Portfolio Value: {cerebro.broker.getvalue():.2f}')

# Run backtest
results = cerebro.run()

# Print final results
print(f'Final Portfolio Value: {cerebro.broker.getvalue():.2f}')

## 5. Analyze Results

Let's examine the performance metrics:

In [ ]:
# Get the first strategy
strat = results[0]

# Print analyzer results
print("\n=== Performance Metrics ===")
print(f"Sharpe Ratio: {strat.analyzers.sharpe.get_analysis().get('sharperatio', 'N/A')}")
print(f"Total Return: {strat.analyzers.returns.get_analysis()['rtot']:.2%}")
print(f"Max Drawdown: {strat.analyzers.drawdown.get_analysis()['max']['drawdown']:.2%}")

## 6. Visualize Results (Optional)

If you have matplotlib installed, you can plot the results:

In [ ]:
# Plot results (requires matplotlib)
try:
    cerebro.plot(style='candlestick')
except Exception as e:
    print(f"Plotting not available: {e}")
    print("Install matplotlib to enable plotting: pip install matplotlib")

## Next Steps

Congratulations! You've completed your first Backtrader backtest. Here are some things to try:

1. **Modify Parameters**: Try different MA periods (e.g., 5/20, 20/50)
2. **Add Indicators**: Experiment with RSI, MACD, Bollinger Bands
3. **Position Sizing**: Implement different position sizing strategies
4. **Risk Management**: Add stop-loss and take-profit levels
5. **Optimization**: Use Cerebro's optimization features to find best parameters

Check out the other tutorials:
- 02_indicators.ipynb - Working with technical indicators
- 03_position_sizing.ipynb - Advanced position sizing
- 04_optimization.ipynb - Parameter optimization
